# ThreadCraft — Measurement Predictor · Step 1: Data Cleaning

**Run on Kaggle with the CPU accelerator.** No GPU needed here or in step 2. Use
**Save & Run All (Commit)** so it runs headlessly.

**Source:** **ANSUR II** (2012 Anthropometric Survey of US Army Personnel), public release
subset — **6,068 people × 93 body measurements**, published by the Penn State OPEN Design Lab.

```
https://tools.openlab.psu.edu/publicData/ANSUR_II_MALE_Public.csv    (4,082 rows)
https://tools.openlab.psu.edu/publicData/ANSUR_II_FEMALE_Public.csv  (1,986 rows)
```

**Licence: US Government work, cleared for unlimited public release** — no attribution
requirement, no non-commercial clause, no gating, no login. This is the cleanest licence of any
dataset in the project.

> ⚠️ **Use `https://`, not `http://`.** The `http://` versions of these URLs now silently serve
> an unrelated HTML page instead of the CSV — you get a 200 response and a file that isn't data.
> The notebook asserts on row count to catch this.

## What this model is for

The proposal names this as an explicit limitation of ThreadCraft:

> *"Measurement accuracy: The platform relies on self-reported customer measurements. Inaccurate
> measurements will affect the fit of the final garment."*

This model attacks that limitation directly, with two capabilities built from the same trained
regressors:

1. **Predict** — a customer who has only measured chest and waist gets sensible suggested values
   for sleeve length, inseam, neck, thigh and the rest, pre-filled in wizard Step 4 as editable
   defaults rather than blank boxes.
2. **Validate** — a measurement that is wildly inconsistent with the others (a mistyped digit, cm
   entered as inches) gets flagged before it reaches the tailor and ruins a garment.

Both map onto the existing `measurement_fields` table, which already stores per-garment min/max
ranges — this model replaces a static range check with a personalised one.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────
HF_USERNAME = "your-hf-username"  # <-- CHANGE THIS to your Hugging Face username

CLEANED_REPO_ID = f"{HF_USERNAME}/threadcraft-measurements-cleaned"

MALE_URL = "https://tools.openlab.psu.edu/publicData/ANSUR_II_MALE_Public.csv"
FEMALE_URL = "https://tools.openlab.psu.edu/publicData/ANSUR_II_FEMALE_Public.csv"

VAL_FRACTION = 0.10
TEST_FRACTION = 0.10
RANDOM_SEED = 42
PUSH_TO_HUB = True

In [ ]:
!pip install -q -U datasets huggingface_hub scikit-learn pandas pyarrow

In [ ]:
import os

from huggingface_hub import login

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print(f"Not on Kaggle or secret missing ({e}). Falling back to the HF_TOKEN env var.")
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face.")
elif PUSH_TO_HUB:
    raise RuntimeError("No HF_TOKEN available but PUSH_TO_HUB is True.")

## 1. Download

Two real gotchas handled here:
- The files are **latin-1** encoded, not UTF-8 — `pd.read_csv` defaults will raise
- The male file's ID column is `subjectid` and the female file's is **`SubjectId`** — different
  case. Concatenating without harmonising silently produces two half-empty ID columns.

In [ ]:
import pandas as pd

male = pd.read_csv(MALE_URL, encoding="latin-1")
female = pd.read_csv(FEMALE_URL, encoding="latin-1")

print(f"male   {male.shape}")
print(f"female {female.shape}")

# Guards against the http:// failure mode, where you get a 200 and an HTML page.
assert male.shape[0] == 4082, f"expected 4082 male rows, got {male.shape[0]} — check the URL is https"
assert female.shape[0] == 1986, f"expected 1986 female rows, got {female.shape[0]}"
print("\nRow counts match the published ANSUR II public release.")

In [ ]:
# Harmonise column case (male: 'subjectid', female: 'SubjectId') and record sex.
male.columns = [c.lower() for c in male.columns]
female.columns = [c.lower() for c in female.columns]
male["sex"] = 1  # 1 = male, 0 = female
female["sex"] = 0

shared = [c for c in male.columns if c in female.columns]
raw = pd.concat([male[shared], female[shared]], ignore_index=True)
print(f"combined: {raw.shape}  ({len(shared)} shared columns)")
print(f"sex balance: {raw['sex'].value_counts().to_dict()}  (1=male, 0=female)")

## 2. Units — the critical conversion

ANSUR II stores **all lengths and circumferences in millimetres**, and — easy to miss —
**`weightkg` is kilograms × 10** (its mean is ~855, i.e. 85.5 kg).

ThreadCraft works entirely in **centimetres**, so everything is converted here, once, in the
cleaning notebook. Getting this wrong by a factor of 10 would produce a model that looks like it
trains fine and then suggests a 10 cm sleeve.

In [ ]:
# ThreadCraft wizard field  ->  ANSUR II column
# Only mappings that are genuinely the same measurement are included.
FIELD_MAP = {
    "height": "stature",
    "chest": "chestcircumference",
    "waist": "waistcircumference",
    "hip": "buttockcircumference",
    "shoulder": "biacromialbreadth",
    "sleeve": "sleevelengthspinewrist",
    "collar": "neckcircumference",
    "inseam": "crotchheight",
    "outseam": "waistheightomphalion",
    "thigh": "thighcircumference",
    "calf": "calfcircumference",
    "cuff": "wristcircumference",
    "ankle": "anklecircumference",
    "total_length": "cervicaleheight",  # neck-to-floor: the garment-length reference
}

missing = [c for c in FIELD_MAP.values() if c not in raw.columns]
assert not missing, f"columns absent from ANSUR II: {missing}"
print(f"{len(FIELD_MAP)} measurement fields mapped, all present.")

clean = pd.DataFrame()
for field, ansur_col in FIELD_MAP.items():
    clean[field] = raw[ansur_col] / 10.0  # mm -> cm

clean["weight"] = raw["weightkg"] / 10.0  # stored as kg*10 -> kg
clean["age"] = raw["age"].astype(float)
clean["sex"] = raw["sex"].astype(int)

print(f"\ncleaned shape: {clean.shape}")
clean.head()

In [ ]:
# Sanity-check every converted column lands in a humanly plausible cm/kg range.
# This is what catches a units error before it reaches training.
EXPECTED_RANGES = {
    "height": (140, 210), "chest": (65, 150), "waist": (55, 145), "hip": (70, 140),
    "shoulder": (28, 55), "sleeve": (60, 115), "collar": (28, 55), "inseam": (60, 110),
    "outseam": (85, 130), "thigh": (40, 85), "calf": (25, 55), "cuff": (13, 23),
    "ankle": (17, 32), "total_length": (120, 185), "weight": (35, 160), "age": (17, 90),
}
problems = []
print(f"{'field':14s} {'min':>8s} {'mean':>8s} {'max':>8s}   plausible?")
print("-" * 56)
for field, (lo, hi) in EXPECTED_RANGES.items():
    s = clean[field]
    ok = (s.min() >= lo * 0.75) and (s.max() <= hi * 1.25)
    if not ok:
        problems.append(field)
    print(f"{field:14s} {s.min():8.1f} {s.mean():8.1f} {s.max():8.1f}   {'yes' if ok else 'NO <-- CHECK UNITS'}")

assert not problems, f"implausible ranges (likely a units bug): {problems}"
print("\nAll fields within plausible human ranges — unit conversion is correct.")

## 3. Clean

ANSUR II is a laboratory-grade measured dataset, not self-reported — so unlike the fit dataset
there is very little to repair. The checks below confirm that rather than assume it.

In [ ]:
print("Missing values per column:")
miss = clean.isnull().sum()
print(miss[miss > 0].to_string() if miss.sum() else "  none")

print(f"\nExact duplicate rows: {clean.duplicated().sum()}")

before = len(clean)
clean = clean.dropna().drop_duplicates().reset_index(drop=True)
print(f"Rows: {before:,} -> {len(clean):,}")

In [ ]:
# BMI is a strong single predictor of the circumference measurements, so derive
# it once here rather than expecting the trees to reconstruct it.
clean["bmi"] = (clean["weight"] / (clean["height"] / 100) ** 2).round(2)
print(clean[["height", "weight", "bmi", "age"]].describe().round(2))

## 4. Exploratory figures

In [ ]:
import matplotlib.pyplot as plt

targets = [f for f in FIELD_MAP if f != "height"]
corr = clean[["height", "weight", "bmi", "age", "sex", *targets]].corr()

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr, cmap="copper_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr)), corr.columns, fontsize=8)
ax.set_title("Correlation between body measurements\n(why prediction from a few inputs works)")
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig("measurement_correlations.png", dpi=130)
plt.show()

print("Strongest predictors of each target (|correlation| with height/weight/bmi):")
for t in targets:
    best = corr.loc[t, ["height", "weight", "bmi"]].abs().idxmax()
    print(f"  {t:14s} {best:8s} r={corr.loc[t, best]:+.3f}")

## 5. Split and push

In [ ]:
from sklearn.model_selection import train_test_split

# Stratify on sex so all three splits keep the 67/33 male/female balance.
train_df, temp_df = train_test_split(
    clean, test_size=VAL_FRACTION + TEST_FRACTION, stratify=clean["sex"], random_state=RANDOM_SEED
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=TEST_FRACTION / (VAL_FRACTION + TEST_FRACTION),
    stratify=temp_df["sex"],
    random_state=RANDOM_SEED,
)
for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    pct = part["sex"].mean() * 100
    print(f"{name:11s} {len(part):6,}   male {pct:.1f}%")

In [ ]:
import json

from datasets import Dataset, DatasetDict

dataset_dict = DatasetDict(
    {
        "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
        "validation": Dataset.from_pandas(val_df.reset_index(drop=True)),
        "test": Dataset.from_pandas(test_df.reset_index(drop=True)),
    }
)

with open("field_map.json", "w") as f:
    json.dump(FIELD_MAP, f, indent=2)

dataset_dict

In [ ]:
if PUSH_TO_HUB:
    try:
        dataset_dict.push_to_hub(CLEANED_REPO_ID, private=False)
        from huggingface_hub import HfApi

        HfApi(token=HF_TOKEN).upload_file(
            path_or_fileobj="field_map.json",
            path_in_repo="field_map.json",
            repo_id=CLEANED_REPO_ID,
            repo_type="dataset",
        )
        print(f"Pushed: https://huggingface.co/datasets/{CLEANED_REPO_ID}")
    except Exception as e:
        print(f"PUSH FAILED: {e}")
        for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
            part.to_parquet(f"/kaggle/working/measurements_{name}.parquet")
        print("Saved parquet to /kaggle/working instead — re-push from there.")
else:
    print("PUSH_TO_HUB is False — nothing uploaded.")

## Summary for the dissertation

In [ ]:
print("=" * 66)
print("DATA PREPARATION SUMMARY — measurement predictor")
print("=" * 66)
print("Source        : ANSUR II (2012 Anthropometric Survey of US Army Personnel)")
print("Publisher     : Penn State OPEN Design Lab, public release subset")
print("Licence       : US Government work — unlimited public release")
print(f"Raw rows      : {len(raw):,} ({(raw['sex'] == 1).sum():,} male, {(raw['sex'] == 0).sum():,} female)")
print(f"After cleaning: {len(clean):,}")
print(f"Fields mapped : {len(FIELD_MAP)} ThreadCraft measurement fields")
print("Units         : ANSUR mm -> cm; weightkg (kg*10) -> kg")
print(f"Train/Val/Test: {len(train_df):,} / {len(val_df):,} / {len(test_df):,} (stratified on sex)")
print("=" * 66)
print("\nNext: run 02_train.ipynb (CPU is fine — no GPU needed).")